# Tutorial: calculus with uxarray
(Previously was restricted to only 1D along grid dimension, but now can handle non-grid dimensions the way you would expect!)

In [ ]:
import numpy as np
import xarray as xr
import uxarray as ux

# for visualizations
import holoviews as hv
hv.extension("bokeh")

# style: hide DataArray data values by default (click icon to expand)
xr.set_options(display_expand_data=False);

# Setup:

## Helper for visualization

In [ ]:
def plot_vector_field(u, v, **kwargs):
    """plot a vector field from its two u, v components."""
    mag = np.hypot(u, v)
    angle = np.arctan2(v, u)
    return hv.VectorField(
        (u.uxgrid.face_lon, u.uxgrid.face_lat, angle, mag), **kwargs
    ).opts(magnitude="Magnitude", color="black")

## Helpers for creating data
(Realistically, most users will probably just `ux.open_dataset()` to load real data instead!)

In [ ]:
# --- Constants ---
SCALE = 4.0                    # characteristic length scale to use as baseline.
LON0, LAT0 = 0.0, 30.0         # center not at 0, for fun :)
REGION = 7.0 * SCALE           # patch of 7 sigma x 7 sigma

# --- Grid: global HEALPix, then subset to a regional patch ---
grid_global = ux.Grid.from_healpix(zoom=6)
grid = grid_global.subset.bounding_box(
    lon_bounds=(LON0 - REGION/2, LON0 + REGION/2),
    lat_bounds=(LAT0 - REGION/2, LAT0 + REGION/2),
)
grid.sphere_radius = 1.0  # --> per-radian results. Use physical radius for per-meter.

# --- Helper for creating data ---
x = ux.UxDataArray(grid.face_lon.values - LON0, dims="n_face", uxgrid=grid, name="x")
y = ux.UxDataArray(grid.face_lat.values - LAT0, dims="n_face", uxgrid=grid, name="y")

def gaussian_vortex(xc, yc, sigma, sign, *, names=("u", "v"), x=x, y=y):
    """return u, v components of a gaussian vortex; clockwise when sign=-1.
    u = -sign*(y-yc)*g; v = +sign*(x-xc)*g; g = exp(-r^2 / 2 sigma^2)."""
    xd, yd = x - xc, y - yc   # distance from center
    g = np.exp(-(xd**2 + yd**2) / (2 * sigma**2))
    u = (-sign * yd * g).rename(names[0])
    v = (+sign * xd * g).rename(names[1])
    return u, v

In [ ]:
grid

In [ ]:
grid.plot()

# Basic case: curl of a single vortex

In [ ]:
# --- Make up some data: vortex with a Gaussian velocity field ---
# (Realistically, users will probably load real data with ds = ux.open_dataset(...),
# then use something like u = ds["zonal_wind"], v = ds["meridional_wind"], instead.)
u, v = gaussian_vortex(xc=0.0, yc=0.0, sigma=scale, sign=+1)
u

In [ ]:
data_plot = plot_vector_field(u, v).opts(aspect=1, arrow_heads=True)
data_plot

## Take the curl, and plot results:

In [ ]:
# --- Curl along the grid dimension ---
curl = u.curl(v)
curl

In [ ]:
vmax = np.abs(curl).max().item()

curl_plot = curl.plot(cmap="RdBu_r", clim=(-vmax, vmax), aspect=1)

In [ ]:
curl_plot * data_plot

There is a strong positive value (red) in the middle because the vortex is counterclockwise.

The white ring and light blue regions come from the underlying gaussian;
more realistic vortices may fall off more slowly and just be entirely red.

The curl is NaN in the outer two layers of cells because they aren't surrounded by neighbors with data.
Rather than extrapolating, uxarray reports NaN here.

# Data with more dimensions ("time")

## Make up some data: two vortices, one moving downwards over time

In [ ]:
sigmaA = 1.0 * SCALE        # vortex A: larger
sigmaB = sigmaA / 1.5       # vortex B: smaller (1.5x ratio)

# --- Time axis and vortex paths ---
nt = 31  # number of timesteps for full dataset
t = np.linspace(0, 1, nt)
time = xr.DataArray(t, dims="time", coords={"time": t})

# Vortex A: stationary, middle left.
# Vortex B: starts top-right, moves down, ends bottom-right
xcA = -1.5 * SCALE
xcB = 1.7 * SCALE
ycA = 0.0
ycB = 0.3*REGION * (1 - 2 * time)    # dims (time,)

uA, vA = gaussian_vortex(xcA, ycA, sigmaA, +1)   # counter-clockwise
uB, vB = gaussian_vortex(xcB, ycB, sigmaB, -1)   # clockwise

u = uA + uB
v = vA + vB

u

## Curl at time=0
(This feature has existed in uxarray for a while, same as above!)

In [ ]:
u0, v0 = u.isel(time=0), v.isel(time=0)
curl_t0 = u0.curl(v0)
curl_t0

In [ ]:
data_plot = plot_vector_field(u0, v0).opts(aspect=1, arrow_heads=True)
curl_plot = curl_t0.plot(cmap="RdBu_r", clim=(-vmax, vmax), aspect=1)
curl_plot * data_plot

## Curl at all times
(New feature, same syntax, loops across times like you would expect!)

In [ ]:
curl = u.curl(v)
curl

In [ ]:
# --- compare to a per-slice workaround ---
curl_workaround = ux.concat(
    [u.isel(time=i).curl(v.isel(time=i)) for i in range(nt)], dim="time"
)

# the values are the same:
diff = (curl_workaround - curl)
print('(min, max) of curl_workaround - curl: ', (diff.min().item(), diff.max().item()))
# (checking np.allclose would show False, though, because (NaN == NaN) is False.)

# here's what the workaround array looks like:
curl_workaround

The workaround is easy enough to write for 1 dim, but imagine having multiple extra dims like "levels" and "time" too! Then it becomes more cumbersome. But, the `u.curl(v)` syntax will work for all cases, regardless of extra dims.

In [ ]:
# --- plotting a few points in time:---
ncols = 2
frame_idx = [0, 7, 14, 21]
vmax = np.abs(curl).max().item()

panels = []
for k, i in enumerate(frame_idx):
    shading = curl.isel(time=i).plot(
        cmap="RdBu_r", clim=(-vmax, vmax), colorbar=(k % ncols == ncols - 1),
        title=f"t = {t[i]:.2f}", aspect=1,
    )
    arrows = plot_vector_field(u.isel(time=i), v.isel(time=i))
    panels.append(shading * arrows)

hv.Layout(panels).cols(ncols)

In [ ]:
# --- Animation (with HoloViews' built-in playback controls) ---
vmax = np.abs(curl).max().item()

def frame(i):
    shading = curl.isel(time=i).plot(
        cmap="RdBu_r", clim=(-vmax, vmax), colorbar=True, aspect=1,
    )
    arrows = plot_vector_field(u.isel(time=i), v.isel(time=i))
    return shading * arrows

anim = hv.HoloMap({f"{t[i]:.3f}": frame(i) for i in range(nt)}, kdims="time")

hv.output(holomap="scrubber", fps=10)
anim

## Can also apply other methods like gradient() and divergence(), too!

### Divergence

In [ ]:
div = u.divergence(v)
vmax = np.abs(div).max().item()

# --- Animation (with HoloViews' built-in playback controls) ---
def frame(i):
    shading = div.isel(time=i).plot(
        cmap="RdBu_r", clim=(-vmax, vmax), colorbar=True, aspect=1,
    )
    arrows = plot_vector_field(u.isel(time=i), v.isel(time=i))
    return shading * arrows

anim = hv.HoloMap({f"{t[i]:.3f}": frame(i) for i in range(nt)}, kdims="time")

hv.output(holomap="scrubber", fps=10)
anim

Note that the divergence limits here are much much smaller!

Why do the values decrease as the vortex moves towards the equator? It's probably due to the metric term:

$$ \text{div}((u,v)) = \quad ... \quad - \frac{v \tan{\varphi}}{a} $$

where $\varphi$ is the latitude and $a$ is the sphere radius. **Prompt: can you prove whether it is actually due to the metric term?**

See the docs page for more details! https://uxarray.readthedocs.io/en/latest/user-guide/vector_calculus.html#divergence-f

### Gradient

Gradient takes a scalar field as an input and ouputs zonal and meridional components,
so the visualization is a little different. For now, let's just take the gradient
of the magnitude of the field to see what kind of outputs it gives:

In [ ]:
mag = np.sqrt(u**2 + v**2).rename("magnitude(u,v)")
mag

In [ ]:
grad = mag.gradient()
grad

Now we once again have a vector field and a scalar field,
but this time the calculus result is the vector field instead!
We can plot it in a similar way, but let's change the color and label things
clearly to avoid confusion:

In [ ]:
vmax_mag = np.abs(mag).max().item()

# --- Animation (with HoloViews' built-in playback controls) ---
def frame(i):
    mag_plot = mag.isel(time=i).plot(
        cmap="viridis", clim=(0, vmax_mag), colorbar=True, aspect=1,
        title="colors=magnitude; arrows=gradient",
    )
    zonal_grad_i = grad['zonal_gradient'].isel(time=i)
    merid_grad_i = grad['meridional_gradient'].isel(time=i)
    grad_arrows = plot_vector_field(zonal_grad_i, merid_grad_i).opts(line_color="red", line_width=1.5, scale=0.8)
    return mag_plot * grad_arrows

anim = hv.HoloMap({f"{t[i]:.3f}": frame(i) for i in range(nt)}, kdims="time")

hv.output(holomap="scrubber", fps=10)
anim

# Working with real data
(To-do: add at least one example with real data into this notebook)